### Run inference OpenAI

Required inputs:

file_name = "interactions_output_file.parquet" <- parquet file containing all interactions

directory = "images" <- directory containing interaction images

api = "..." <- your own OpenAI api 

In [ ]:
file_name = "interactions_output_file.parquet" 
directory = "images"
api = "YOUR_OPENAI_API_KEY"

Libraries:

In [ ]:
import base64
import time
from pathlib import Path
import pandas as pd
from openai import OpenAI

In [ ]:
client = OpenAI(api_key=api)

# Use the exact model IDs available in your account.
MODELS = [
    "gpt-5.4-mini", 
    # "gpt-5.4",
]

def image_to_data_url(image_path: str) -> str:
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    suffix = path.suffix.lower()
    mime_map = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
    }
    mime_type = mime_map.get(suffix)
    if mime_type is None:
        raise ValueError(f"Unsupported image type: {suffix}")

    image_bytes = path.read_bytes()
    b64 = base64.b64encode(image_bytes).decode("utf-8")
    return f"data:{mime_type};base64,{b64}"


def ask_gpt(model_name: str, image_path: str, caption: str) -> str:
    image_data_url = image_to_data_url(image_path)

    response = client.responses.create(
        model=model_name,
        temperature = 0.0,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": caption},
                    {"type": "input_image", "image_url": image_data_url},
                ],
            }
        ],
    )

    return response.output_text.strip()


df = pd.read_parquet(file_name,      
                    engine="pyarrow",
                    dtype_backend="pyarrow")
required_cols = ["fileName", "caption"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

results = []

for idx, row in df.iterrows():
    image_path = "/Volumes/Extreme SSD/interactions/inat_images/"+row["fileName"]
    image_path = row["fileName"]
    caption = row["caption"]
    ground_truth = row["ground_truth"] if "ground_truth" in df.columns else None

    print(f"\nRow {idx}")
    print("Image:", image_path)
    print("Caption:", caption)

    for model_name in MODELS:
        try:
            prediction = ask_gpt(
                model_name=model_name,
                image_path=image_path,
                caption=caption,
            )

            result_row = {
                "index": idx,
                "model": model_name,
                "fileName": row["fileName"],
                "caption": caption,
                "prediction": prediction,
                "status": "ok",
            }

            if ground_truth is not None:
                result_row["ground_truth"] = ground_truth

            print(prediction)
            results.append(result_row)
            print(f"[{model_name}] OK")

        except Exception as e:
            result_row = {
                "index": idx,
                "model": model_name,
                "fileName": row["fileName"],
                "caption": caption,
                "prediction": None,
                "status": f"error: {e}",
            }

            if ground_truth is not None:
                result_row["ground_truth"] = ground_truth

            results.append(result_row)
            print(f"[{model_name}] ERROR: {e}")

        time.sleep(0.5)

results_df = pd.DataFrame(results)
results_df.to_csv("openai_inference_results.csv", index=False)

print("\nDone.")
print(results_df.head())